In [ ]:
# filename: run_tesla_lstm.py
import os
import random
import math
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from typing import Callable, Dict, List, Tuple
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ------------------------------
# Reproducibility
# ------------------------------
def set_seed(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

# ------------------------------
# Data utilities
# ------------------------------
def load_prices(ticker: str = "TSLA", period: str = "10y", interval: str = "1d") -> pd.Series:
    df = yf.download(ticker, period=period, interval=interval, auto_adjust=True)
    return df["Close"].dropna()

def make_supervised(series: np.ndarray, window: int = 60) -> Tuple[np.ndarray, np.ndarray]:
    X, y = [], []
    for i in range(window, len(series)):
        X.append(series[i-window:i])
        y.append(series[i])
    X = np.array(X)
    y = np.array(y)
    return X.reshape(X.shape[0], X.shape[1], 1), y

def train_val_test_split(values: np.ndarray, train_ratio=0.7, val_ratio=0.15):
    n = len(values)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train, val, test = values[:n_train], values[n_train:n_train+n_val], values[n_train+n_val:]
    return train, val, test

# ------------------------------
# Model builders (LSTM variants)
# ------------------------------
def lstm_basic(window: int, units=50, dropout=0.1):
    model = Sequential([
        LSTM(units, input_shape=(window,1)),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def lstm_stacked(window: int, units=50, dropout=0.2):
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=(window,1)),
        Dropout(dropout),
        LSTM(units),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def lstm_bidirectional(window: int, units=50, dropout=0.2):
    from tensorflow.keras.layers import Bidirectional
    model = Sequential([
        Bidirectional(LSTM(units), input_shape=(window,1)),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def get_model_zoo(window: int) -> Dict[str, Callable[[], tf.keras.Model]]:
    return {
        "LSTM_Basic": lambda: lstm_basic(window),
        "LSTM_Stacked": lambda: lstm_stacked(window),
        "LSTM_Bidirectional": lambda: lstm_bidirectional(window),
    }

# ------------------------------
# Training + evaluation
# ------------------------------
def inverse_transform(scaler, arr):
    return scaler.inverse_transform(arr.reshape(-1,1)).flatten()

def evaluate(y_true, y_pred) -> Dict[str, float]:
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-8, None))) * 100
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "MAPE": mape}

def train_and_evaluate(name, builder, X_train, y_train, X_val, y_val, X_test, y_test, scaler, epochs=50, batch_size=64):
    model = builder()
    cb = [
        EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
    ]
    history = model.fit(X_train, y_train,
                        validation_data=(X_val, y_val),
                        epochs=epochs,
                        batch_size=batch_size,
                        callbacks=cb,
                        verbose=0)

    preds = model.predict(X_test, verbose=0).flatten()

    # inverse scale
    y_test_inv = inverse_transform(scaler, y_test)
    preds_inv = inverse_transform(scaler, preds)

    metrics = evaluate(y_test_inv, preds_inv)

    return {"name": name, "model": model, "history": history.history,
            "y_test": y_test_inv, "y_pred": preds_inv, "metrics": metrics}

# ------------------------------
# Plot helpers
# ------------------------------
def plot_predictions(result):
    plt.figure(figsize=(10,5))
    plt.plot(result["y_test"], label="Actual")
    plt.plot(result["y_pred"], label="Predicted")
    plt.title(f"{result['name']} - Test Predictions")
    plt.legend()
    plt.show()

def plot_val_losses(results):
    plt.figure(figsize=(10,5))
    for r in results:
        plt.plot(r["history"]["val_loss"], label=r["name"])
    plt.title("Validation Loss")
    plt.legend()
    plt.show()

# ------------------------------
# Main experiment
# ------------------------------
def run_experiment(ticker="TSLA", window=60, epochs=50, batch_size=64):
    print(f"Downloading {ticker} data...")
    close = load_prices(ticker)

    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(close.values.reshape(-1,1)).flatten()

    train_vals, val_vals, test_vals = train_val_test_split(scaled)
    X_train, y_train = make_supervised(train_vals, window)
    X_val, y_val     = make_supervised(val_vals, window)
    X_test, y_test   = make_supervised(test_vals, window)

    zoo = get_model_zoo(window)
    results = []
    for name, builder in zoo.items():
        print(f"Training {name}...")
        r = train_and_evaluate(name, builder, X_train, y_train, X_val, y_val, X_test, y_test, scaler, epochs, batch_size)
        results.append(r)

    # Print metrics
    print("\n=== Evaluation Results ===")
    for r in results:
        print(f"{r['name']}: {r['metrics']}")

    plot_val_losses(results)
    for r in results:
        plot_predictions(r)

    return results

if __name__ == "__main__":
    run_experiment()
